# Ejercitación · Soluciones paso a paso en R

Taller 1 · Iván Canales · UTFSM

Este notebook resuelve el desafío de programación y las diez preguntas estadísticas del apartado [Ejercitación](https://ivanrenatoc.github.io/UTFSM-econometria/#ejercitacion). Todo el código es **R**. Ejecuta las celdas de arriba hacia abajo con un **kernel R** en Jupyter o un entorno de Colab configurado para R. La extensión correcta del archivo es `.ipynb`.

Cada respuesta sigue cuatro pasos: definir qué se busca, preparar los datos necesarios, calcular o graficar y redactar una interpretación con unidades y límites. Las celdas generan los valores y frases de respuesta al ejecutarse; no contienen resultados estadísticos precalculados.

**Estado de validación:** código preparado y estructura del notebook verificada; R no está disponible en el entorno de elaboración, por lo que las celdas aún no se han ejecutado. La instalación inicial requiere conexión a Internet.

## Parte 1 · Programación en R: optimizar 100 wall balls

En disciplinas deportivas y clubes de entrenamiento se practica un ejercicio llamado wall ball. Consiste en lanzar un balón de aproximadamente 60 cm de diámetro mientras se pasa de una posición de sentadilla a una postura completamente erguida.

Para este problema, supón que una persona debe completar 100 lanzamientos y no puede realizar más de 20 lanzamientos seguidos y debe completar la rutina en un máximo de 20 tandas. Una tanda es un grupo de lanzamientos consecutivos antes de descansar. Cada vez que interrumpe una tanda para descansar, el descanso dura tres segundos por cada lanzamiento de esa tanda: por ejemplo, después de cinco lanzamientos descansa 15 segundos. No se cuenta un descanso después del lanzamiento número 100.

Dentro de cada tanda, el lanzamiento número j tarda 1,1 + 0,3 × j segundos. La numeración comienza en j = 1 y se reinicia después de cada descanso. Supón que la persona puede retomar el ejercicio con ese mismo patrón tras descansar.

Escribe un programa en R que utilice ciclos for o while y vectores para encontrar una combinación de tandas que permita completar los 100 lanzamientos en el menor tiempo total posible. Almacena la cantidad de lanzamientos de cada tanda, calcula sus tiempos de ejecución y descanso, y muestra la sucesión óptima y el tiempo total. Justifica que la solución es óptima, en lugar de comparar solo algunas combinaciones elegidas a mano.

El diámetro y las reglas de tiempo son supuestos del ejercicio de programación. No se incluye un tiempo fijo adicional por detenerse ni por reiniciar una tanda.

### Paso 1 · Diferenciar los dos límites

Hay 100 lanzamientos en total, como máximo 20 lanzamientos por tanda y como máximo 20 tandas en toda la rutina. El descanso se realiza entre tandas, nunca después de la última. El contador de repetición j se reinicia en cada tanda.

### Paso 2 · Guardar el mejor tiempo por lanzamientos y tandas

Ahora el estado debe registrar dos cantidades: n lanzamientos pendientes y b tandas exactas. Guardamos los estados en un vector, con la función posicion(n,b). Inicializamos solo el caso (0,0) con tiempo cero; los estados imposibles tienen tiempo infinito.

Para cada estado probamos la primera tanda k entre 1 y min(20,n). Su costo es el tiempo de lanzamiento, más 3*k segundos de descanso si quedan lanzamientos, más el mejor tiempo del estado (n-k,b-1). El ciclo compara todas las primeras tandas posibles. Así limita tanto las repeticiones como la cantidad de tandas.

Finalmente comparamos los tiempos de completar 100 lanzamientos en 1, 2, …, 20 tandas. No suponemos de antemano que deben usarse las 20: lo determina el mínimo. Se trabaja en décimas de segundo para mantener comparaciones exactas.

**Por qué es óptimo:** toda solución con b tandas empieza con algún k permitido y continúa con b−1 tandas. Reemplazar ese resto por su mejor solución no puede empeorar el tiempo. Al evaluar todos los k y luego todas las cantidades b permitidas, se consideran todos los planes factibles.

In [ ]:
# Restricciones: 100 lanzamientos, como máximo 20 tandas,
# y entre 1 y 20 lanzamientos en cada tanda.
meta <- 100
max_tandas <- 20
max_lanzamientos <- 20
# Tiempos en décimas de segundo: evita errores de redondeo.
tiempo_tanda <- numeric(max_lanzamientos)
for (k in 1:max_lanzamientos) {
  for (j in 1:k) {
    tiempo_tanda[k] <- tiempo_tanda[k] + 11 + 3 * j
  }
}

# Guardamos en vectores los estados (n lanzamientos, b tandas exactas).
# Cada grupo de meta+1 posiciones corresponde a una cantidad de tandas.
posicion <- function(n, b) {
  n + 1 + b * (meta + 1)
}
mejor <- rep(Inf, (meta + 1) * (max_tandas + 1))
elegida <- numeric(length(mejor))
mejor[posicion(0, 0)] <- 0
for (b in 1:max_tandas) {
  for (n in 1:meta) {
    for (k in 1:min(max_lanzamientos, n)) {
      descanso <- 0
      if (k < n) {
        descanso <- 30 * k
      }
      candidato <- tiempo_tanda[k] + descanso + mejor[posicion(n - k, b - 1)]
      if (candidato < mejor[posicion(n, b)]) {
        mejor[posicion(n, b)] <- candidato
        elegida[posicion(n, b)] <- k
      }
    }
  }
}
# Elegir entre todas las cantidades permitidas de tandas, no imponer 20.
tiempo_final <- numeric(max_tandas)
for (b in 1:max_tandas) {
  tiempo_final[b] <- mejor[posicion(meta, b)]
}
if (!any(is.finite(tiempo_final))) {
  stop("No existe un plan que cumpla estas restricciones.")
}
b_optimo <- which.min(tiempo_final)

# Recuperar la sucesión óptima.
tandas <- numeric(b_optimo)
pendientes <- meta
b_restantes <- b_optimo
i <- 0
while (pendientes > 0) {
  i <- i + 1
  k <- elegida[posicion(pendientes, b_restantes)]
  tandas[i] <- k
  pendientes <- pendientes - k
  b_restantes <- b_restantes - 1
}
ejecucion <- tiempo_tanda[tandas] / 10
descansos <- 3 * tandas
descansos[length(tandas)] <- 0
registro <- data.frame(
  tanda = seq_along(tandas), lanzamientos = tandas,
  acumulado = cumsum(tandas), ejecucion_seg = ejecucion,
  descanso_seg = descansos,
  tiempo_acumulado_seg = cumsum(ejecucion + descansos)
)
print(registro)
print(tandas)
cat("Tiempo mínimo:", tiempo_final[b_optimo] / 10, "segundos\n")
cat("Tandas:", length(tandas), "| Descansos:", length(tandas) - 1, "\n")
cat("Tiempo lanzando:", sum(ejecucion), "| Tiempo descansando:", sum(descansos), "\n")
stopifnot(sum(tandas) == meta, length(tandas) <= max_tandas,
          all(tandas >= 1 & tandas <= max_lanzamientos),
          b_restantes == 0,
          abs(sum(ejecucion + descansos) - tiempo_final[b_optimo] / 10) < 1e-9)


### Paso 3 · Recuperar y comprobar la solución

El while consulta la primera tanda elegida para los lanzamientos y las tandas que quedan. En cada iteración disminuyen ambas cantidades. El registro muestra cada tanda, acumulados, tiempos de lanzamiento y descansos. Las comprobaciones finales exigen 100 lanzamientos, no más de 20 tandas y no más de 20 lanzamientos por tanda.

El mínimo es **471.5 segundos**, con **20 tandas y 19 descansos**. Una sucesión óptima es:

`4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 15`

Son **216.5 segundos lanzando** y **255.0 segundos descansando**. El orden de tandas anteriores a la última puede variar sin cambiar el tiempo si conserva sus tamaños; no se afirma que el plan sea único.

### Paso 4 · Redactar una buena respuesta

«Busqué el mínimo entre todos los planes con hasta 20 tandas, respetando el máximo de 20 lanzamientos por tanda. Guardé los mejores tiempos y las elecciones en vectores y recuperé el plan mediante un while. La sucesión mostrada suma 100, cumple los límites y su tiempo coincide con el mínimo calculado».

La solución anterior con 91 tandas era inadmisible bajo el límite de 20 tandas. Agregar esa restricción cambia el problema: no basta con conservar solo el mejor tiempo por número de lanzamientos. Es necesario recordar también cuántas tandas se utilizan.

## Parte 2 · Preparación de wage2

Usamos otra base transversal del libro Wooldridge. El [diccionario oficial de wage2](https://justinmshea.github.io/wooldridge/reference/wage2.html) documenta 935 observaciones, 17 variables y datos de 1980. `wage` es **ingreso mensual**, no ingreso por hora. Cada fila corresponde a una persona.

`educ`, `exper` y `tenure` se miden en años; `age` es edad en años; `hours`, horas semanales promedio; `married` vale 1 si está casado y 0 si no está casado. «No casado» no significa exclusivamente «soltero». No se utiliza `female`, porque no es una variable de esta base.

### Paso inicial · Instalar si hace falta y cargar una sola vez

La instalación es condicional; en ejecuciones posteriores se utiliza el paquete ya instalado. Conservamos la base original y preparamos subconjuntos específicos por pregunta para no eliminar filas por faltantes en variables ajenas a cada cálculo.

In [ ]:
if (!requireNamespace("wooldridge", quietly = TRUE)) {
  install.packages("wooldridge", repos = "https://cloud.r-project.org/")
}
datos <- as.data.frame(wooldridge::wage2)
variables <- c("wage", "educ", "exper", "tenure", "age", "hours", "married")
colores <- c("#0160a0", "#f8c751", "#00875a")
options(digits = 6)
head(datos[variables])


## Pregunta 1 · Inspecciona la base: ¿cuántas filas y columnas contiene y qué tipo de datos son? Cuenta los faltantes de las siete variables seleccionadas. ¿Qué representa una fila y en qué unidad se mide wage?

**Paso 1:** usar `dim()` y `str()` para distinguir dimensiones y tipos. **Paso 2:** contar faltantes por variable. **Paso 3:** documentar la unidad de observación y las advertencias del diccionario.

In [ ]:
print(dim(datos))
str(datos)
print(data.frame(variable = variables,
                 faltantes = colSums(is.na(datos[variables]))))
cat(sprintf("La base contiene %d personas y %d variables. Cada fila es una persona; wage es ingreso mensual.
",
            nrow(datos), ncol(datos)))
# Señales que merecen revisión, no corrección automática:
print(c(tenure_mayor_que_exper = sum(datos$tenure > datos$exper, na.rm = TRUE),
        exper_mayor_que_edad = sum(datos$exper > datos$age, na.rm = TRUE)))


**Cómo completar la respuesta:** indica que son datos de corte transversal correspondientes a 1980 y copia los conteos observados. La documentación advierte inconsistencias entre antigüedad, experiencia y edad. Las dos comparaciones del código son señales de revisión, no una auditoría exhaustiva ni una justificación para borrar registros. No inventes un criterio de corrección sin información adicional.

## Pregunta 2 · Calcula la media, mediana, moda (incluye todas si hay empates) y percentiles 24, 50 y 75 de wage. ¿Qué revela la diferencia entre media y mediana?

**Paso 1:** conservar salarios observados e informar exclusiones. **Paso 2:** calcular medidas de posición. **Paso 3:** construir una tabla de frecuencias para la moda: `mode()` en R describe el tipo de un objeto, no la moda estadística. **Paso 4:** comparar media y mediana sin tratar esa comparación como una prueba definitiva de asimetría.

In [ ]:
salarios <- datos$wage[!is.na(datos$wage)]
n_salarios <- length(salarios)
cat("Casos utilizados:", n_salarios, "| Faltantes excluidos:", sum(is.na(datos$wage)), "
")
frecuencias <- table(salarios)
modas <- as.numeric(names(frecuencias)[frecuencias == max(frecuencias)])
posicion <- c(media = mean(salarios), mediana = median(salarios),
              quantile(salarios, probs = c(0.24, 0.50, 0.75), type = 7))
print(posicion)
print(data.frame(moda = modas, frecuencia = as.integer(max(frecuencias))))
cat(sprintf("El ingreso mensual medio es %.2f y la mediana %.2f; la diferencia es %.2f unidades monetarias mensuales.
",
            mean(salarios), median(salarios), mean(salarios) - median(salarios)))


**Interpretación:** la mediana y P50 coinciden. P24 deja aproximadamente el 24 % de las observaciones a su izquierda y P75, el 75 %; con datos discretos y empates la fracción exacta puede diferir. Si la media supera la mediana, es compatible con una cola hacia ingresos altos; confírmalo con el histograma. Informa todas las modas si hay empate. Los percentiles usan la interpolación `type = 7`, predeterminada en R.

## Pregunta 3 · Calcula varianza, desviación estándar, rango intercuartílico y coeficiente de variación de wage. Explica las unidades de cada medida y expresa el coeficiente de variación como porcentaje.

**Paso 1:** reutilizar los salarios válidos de la pregunta anterior. **Paso 2:** calcular medidas de dispersión. **Paso 3:** distinguir escala y unidades. R usa divisor `n − 1` para la varianza muestral.

In [ ]:
dispersion <- c(varianza = var(salarios),
                desviacion_estandar = sd(salarios),
                rango_intercuartilico = IQR(salarios, type = 7),
                coeficiente_variacion_porcentaje = 100 * sd(salarios) / mean(salarios))
print(dispersion)
cat("Casos utilizados:", n_salarios, "| Faltantes excluidos:", nrow(datos) - n_salarios, "
")
cat(sprintf("La desviación estándar es %.2f unidades monetarias mensuales y el CV es %.2f %%.
",
            sd(salarios), 100 * sd(salarios) / mean(salarios)))


**Interpretación:** varianza en unidades monetarias mensuales **al cuadrado**; desviación estándar e IQR en la misma unidad del ingreso. El IQR mide la amplitud del 50 % central. El CV no tiene unidades y compara la desviación con la media; aquí se expresa en porcentaje. Es adecuado para una escala con cero significativo y media positiva; no es una probabilidad ni el porcentaje de personas con salarios «variables».

## Pregunta 4 · Dibuja un histograma y un boxplot de wage, con títulos y unidades. Describe la asimetría y los posibles valores atípicos; explica por qué un valor extremo no debe eliminarse automáticamente.

**Paso 1:** representar la distribución sin borrar extremos. **Paso 2:** usar el histograma para forma y el boxplot para posición y dispersión. **Paso 3:** identificar los puntos fuera de los bigotes y revisar su contexto.

In [ ]:
op <- par(mfrow = c(1, 2))
hist(salarios, breaks = "FD", col = colores[1], border = "white",
     main = "Distribución del ingreso mensual",
     xlab = "Ingreso mensual (unidad monetaria de wage2)", ylab = "Personas")
boxplot(salarios, col = colores[2],
        main = "Ingreso mensual: mediana y dispersión",
        ylab = "Ingreso mensual (unidad monetaria de wage2)")
par(op)
atipicos <- boxplot.stats(salarios)$out
cat("Casos utilizados:", n_salarios, "| Faltantes excluidos:", nrow(datos) - n_salarios, "
")
cat("Observaciones señaladas fuera de los bigotes:", length(atipicos), "
")
print(sort(atipicos))


**Cómo redactar:** describe dónde se concentra la mayoría de los ingresos, hacia qué lado se extiende la cola y si aparecen puntos fuera de los bigotes. En el boxplot estándar, los bigotes se extienden hasta observaciones dentro de 1,5 IQR de las bisagras. Los puntos externos son candidatos a revisión; pueden ser ingresos reales, no necesariamente errores. No los elimines solo porque el gráfico los marca.

## Pregunta 5 · Compara el ingreso mensual de casados y no casados: muestra tamaños de grupo, medias, medianas y un boxplot por married. ¿En qué difieren sus distribuciones?

**Paso 1:** conservar casos con salario y estado civil observados. **Paso 2:** separar 0 y 1 sin confundir las etiquetas. **Paso 3:** comparar números y distribuciones; no limitar la respuesta a cuál media es mayor.

In [ ]:
validos_estado <- complete.cases(datos[c("wage", "married")])
estado <- datos[validos_estado, c("wage", "married")]
stopifnot(all(estado$married %in% c(0, 1)))
estado$grupo <- factor(estado$married, levels = c(0, 1), labels = c("No casado", "Casado"))
por_grupo <- split(estado$wage, estado$grupo)
resumen_grupos <- do.call(rbind, lapply(por_grupo, function(x) {
  c(n = length(x), media = mean(x), mediana = median(x), desviacion = sd(x))
}))
print(resumen_grupos)
cat("Casos utilizados:", nrow(estado), "| Faltantes excluidos:", sum(!validos_estado), "
")
boxplot(wage ~ grupo, data = estado, col = colores[1:2],
        main = "Ingreso mensual por estado civil", xlab = "Estado civil",
        ylab = "Ingreso mensual (unidad monetaria de wage2)")
cat(sprintf("La diferencia descriptiva de medias (casado menos no casado) es %.2f.
",
            mean(por_grupo[["Casado"]]) - mean(por_grupo[["No casado"]])))


**Una buena respuesta:** comunica tamaños de grupo, media y mediana y describe el solapamiento, la dispersión y los extremos del gráfico. Una diferencia de medias no implica que cada persona de un grupo gane más que cada persona del otro, ni identifica el efecto causal del matrimonio.

## Pregunta 6 · Calcula las correlaciones de Pearson entre wage, educ, exper y tenure usando los mismos casos completos. Dibuja wage frente a educ y describe la asociación, sin atribuirle causalidad.

**Paso 1:** fijar una única muestra completa para las cuatro variables. Así todas las correlaciones se calculan sobre las mismas personas. **Paso 2:** calcular Pearson. **Paso 3:** mirar el diagrama de dispersión antes de interpretar el coeficiente.

In [ ]:
variables_cor <- c("wage", "educ", "exper", "tenure")
validos_cor <- complete.cases(datos[variables_cor])
datos_cor <- datos[validos_cor, variables_cor]
print(round(cor(datos_cor, method = "pearson"), 3))
cat("Casos utilizados:", nrow(datos_cor), "| Faltantes excluidos:", sum(!validos_cor), "
")
plot(datos_cor$educ, datos_cor$wage, pch = 19,
     col = adjustcolor(colores[1], alpha.f = 0.35),
     xlab = "Educación (años)", ylab = "Ingreso mensual (unidad monetaria de wage2)",
     main = "Educación e ingreso mensual")
cat(sprintf("La correlación de Pearson entre educación e ingreso mensual es %.3f.
",
            cor(datos_cor$educ, datos_cor$wage)))


**Interpretación:** el signo indica la dirección de la asociación lineal y la magnitud su intensidad. Pearson está entre −1 y 1, es sensible a extremos y no resume bien todas las relaciones no lineales. Describe también la variación de salarios para un mismo nivel educativo. Una correlación no controla experiencia, antigüedad ni otros factores; no demuestra causalidad.

## Pregunta 7 · Construye un intervalo aproximado de confianza del 95 % para el ingreso mensual medio: media ± qnorm(0.975) × sd(wage)/sqrt(n), con casos válidos. Justifica la aproximación de muestra grande e interpreta el intervalo como estimación de una media, no de salarios individuales.

**Paso 1:** definir el parámetro: ingreso mensual medio de la población de referencia. **Paso 2:** estimar el error estándar `s/√n`. **Paso 3:** aplicar el valor normal crítico del 95 %, como solicita el ejercicio. La desviación poblacional no es conocida: usamos la muestral en una aproximación de muestra grande, no un intervalo normal exacto.

In [ ]:
media <- mean(salarios)
s <- sd(salarios)
n <- length(salarios)
error_estandar <- s / sqrt(n)
z <- qnorm(0.975)
margen <- z * error_estandar
ic_media <- c(inferior = media - margen, superior = media + margen)
print(c(n = n, faltantes_excluidos = nrow(datos) - n,
        media = media, error_estandar = error_estandar, margen = margen, ic_media))
cat(sprintf("El intervalo aproximado del 95 %% para la media mensual es [%.2f, %.2f].
",
            ic_media[1], ic_media[2]))


**Cómo justificar y responder:** el tamaño grande permite usar una aproximación normal de la media, bajo independencia y condiciones del TCL (varianza finita y ausencia de observaciones que dominen la suma). El tamaño no corrige un sesgo de selección. Si se repitiera el procedimiento con muestras comparables bajo sus supuestos, alrededor del 95 % de los intervalos contendrían la media poblacional. No afirma que el 95 % de las personas gane dentro de estos límites. Con σ desconocida también puede usarse un intervalo t; aquí se mantiene la aproximación normal solicitada.

## Pregunta 8 · Estima la proporción de personas casadas y su intervalo normal aproximado del 95 %: p ± qnorm(0.975) × sqrt(p*(1-p)/n). Comprueba n ≥ 30, éxitos > 5 y fracasos > 5 antes de usarlo. ¿Qué población podrías describir y bajo qué supuestos?

**Paso 1:** usar todos los estados civiles observados, sin exigir salario. **Paso 2:** contar casados y no casados. **Paso 3:** revisar los requisitos antes de formar el intervalo normal. **Paso 4:** presentar proporción e intervalo como porcentajes.

In [ ]:
m <- datos$married[!is.na(datos$married)]
stopifnot(all(m %in% c(0, 1)))
n_prop <- length(m)
exitos <- sum(m == 1)
fracasos <- sum(m == 0)
p <- exitos / n_prop
print(c(n = n_prop, faltantes_excluidos = sum(is.na(datos$married)),
        casados = exitos, no_casados = fracasos, proporcion = p))
if (n_prop >= 30 && exitos > 5 && fracasos > 5) {
  se_prop <- sqrt(p * (1 - p) / n_prop)
  ic_prop <- p + c(-1, 1) * qnorm(0.975) * se_prop
  print(c(inferior = ic_prop[1], superior = ic_prop[2]))
  cat(sprintf("Se observa %.2f %% de personas casadas; el IC aproximado del 95 %% es [%.2f %%, %.2f %%].
",
              100 * p, 100 * ic_prop[1], 100 * ic_prop[2]))
} else {
  cat("No se cumplen los requisitos del intervalo normal solicitado. No lo interpretamos.
")
}


**Interpretación:** la proporción observada describe estas personas. Solo puede extenderse a la población de origen si el diseño y la selección sustentan representatividad e independencia. No generalices a Chile actual ni a toda la población mundial. El intervalo normal de Wald es una aproximación y puede fallar cerca de 0 o 1; no se arregla recortando arbitrariamente los extremos a [0,1].

## Pregunta 9 · Estima la diferencia de medias de wage (casados menos no casados) y un intervalo aproximado del 95 %: diferencia ± qnorm(0.975) × sqrt(s1^2/n1 + s0^2/n0). Informa los tamaños de ambos grupos, revisa la aproximación de muestras grandes e indica si el intervalo contiene cero. ¿Por qué no demuestra un efecto causal del matrimonio?

**Paso 1:** recuperar las mismas personas completas de la pregunta 5. **Paso 2:** fijar el orden de la resta: casados menos no casados. **Paso 3:** combinar los errores de las dos medias suponiendo grupos independientes; no se asumen varianzas iguales. **Paso 4:** revisar si el intervalo contiene cero.

In [ ]:
x1 <- por_grupo[["Casado"]]
x0 <- por_grupo[["No casado"]]
n1 <- length(x1)
n0 <- length(x0)
print(c(n_casados = n1, n_no_casados = n0,
        faltantes_excluidos = nrow(datos) - n1 - n0))
if (n1 >= 30 && n0 >= 30) {
  diferencia <- mean(x1) - mean(x0)
  se_diferencia <- sqrt(var(x1) / n1 + var(x0) / n0)
  ic_diferencia <- diferencia + c(-1, 1) * qnorm(0.975) * se_diferencia
  contiene_cero <- ic_diferencia[1] <= 0 && ic_diferencia[2] >= 0
  print(c(diferencia = diferencia, error_estandar = se_diferencia,
          inferior = ic_diferencia[1], superior = ic_diferencia[2]))
  cat(sprintf("La diferencia estimada es %.2f; IC aproximado del 95 %%: [%.2f, %.2f].
",
              diferencia, ic_diferencia[1], ic_diferencia[2]))
  cat(if (contiene_cero) "El intervalo contiene cero: no detectamos diferencia al nivel elegido.
"
      else "El intervalo excluye cero: hay evidencia de una diferencia bajo los supuestos.
")
} else {
  cat("Revisar la aproximación: uno de los grupos tiene menos de 30 observaciones.
")
}


**Interpretación:** los límites se expresan en unidades de ingreso mensual. El chequeo de 30 casos por grupo es orientativo; revisa también asimetría y extremos. La independencia de personas y de los grupos no puede probarse solo con estos conteos. Incluir cero no prueba igualdad; excluirlo no prueba que el matrimonio cause la diferencia. Educación, experiencia y selección pueden explicar parte de la asociación. Si se usara inferencia t con varianzas desconocidas y diferentes, correspondería la variante de Welch; este ejercicio pide la aproximación normal.

## Pregunta 10 · Compara los intervalos del 90 %, 95 % y 99 % para la media total de wage en un gráfico con un punto central y una barra por intervalo. ¿Cómo cambia su amplitud al aumentar la confianza? Si cuadruplicaras n manteniendo la dispersión, ¿qué pasaría con el margen?

**Paso 1:** mantener los mismos datos, media y error estándar de la pregunta 7. **Paso 2:** cambiar únicamente el nivel de confianza. **Paso 3:** dibujar un punto central y una barra horizontal por intervalo. **Paso 4:** separar el efecto de la confianza del efecto del tamaño muestral.

In [ ]:
niveles <- c(0.90, 0.95, 0.99)
criticos <- qnorm(1 - (1 - niveles) / 2)
margenes <- criticos * error_estandar
comparacion <- data.frame(confianza = niveles, n = n,
                          media = media, inferior = media - margenes,
                          superior = media + margenes, margen = margenes,
                          amplitud = 2 * margenes)
print(comparacion)
cat("Casos utilizados:", n, "| Faltantes excluidos:", nrow(datos) - n, "
")
posiciones <- seq_along(niveles)
plot(rep(media, length(niveles)), posiciones,
     xlim = range(c(comparacion$inferior, comparacion$superior)),
     ylim = c(0.5, 3.5), yaxt = "n", pch = 19, col = colores,
     xlab = "Ingreso mensual medio (unidad monetaria de wage2)", ylab = "Confianza",
     main = "Mayor confianza, mayor amplitud")
axis(2, at = posiciones, labels = paste0(100 * niveles, " %"), las = 1)
segments(comparacion$inferior, posiciones, comparacion$superior, posiciones,
         col = colores, lwd = 3)
points(rep(media, length(niveles)), posiciones, pch = 19, col = colores)
abline(v = media, lty = 3, col = "gray50")
margen_actual <- qnorm(0.975) * s / sqrt(n)
margen_cuatro_n <- qnorm(0.975) * s / sqrt(4 * n)
print(c(margen_95_actual = margen_actual,
        margen_95_con_4n = margen_cuatro_n,
        razon = margen_cuatro_n / margen_actual))


**Respuesta final:** al mantener muestra y dispersión, el centro de los tres intervalos es el mismo. La mayor confianza aumenta el valor crítico y ensancha el intervalo. Si se cuadruplica `n` manteniendo confianza y dispersión, el margen se divide por dos porque `√(4n) = 2√n`. Es una comparación teórica: el código no crea nuevas personas ni demuestra que una muestra real cuatro veces mayor conserve exactamente la misma dispersión.

## Revisión antes de entregar

- Ejecuta desde una sesión limpia y conserva las salidas y los gráficos.
- Comprueba que cada respuesta incluya cálculo, casos utilizados, faltantes y unidades.
- Redacta conclusiones con los valores que obtuviste, no solo con nombres de funciones.
- Separa descripción, incertidumbre y causalidad.
- No reemplaces faltantes por ceros ni elimines extremos sin justificarlo.

Fuente de variables y contexto: [documentación oficial de wage2](https://justinmshea.github.io/wooldridge/reference/wage2.html). Consignas: [taller de Econometría UTFSM](https://ivanrenatoc.github.io/UTFSM-econometria/#ejercitacion).

In [ ]:
sessionInfo()
